In [ ]:
import numpy as np
import itertools
from mat_lib_build import FuelSpecification,build_fuel_material

In [ ]:
def generate_fuel_combos(axes):
    gad_concs = np.asanyarray(axes["gad_conc"])

    # 1. Standard Fuel Stream (gad_conc ~ 0.0)
    if np.any(np.isclose(gad_concs, 0.0)):
        for uenrich, (temp, xs) in itertools.product(axes["uenrich"], axes["temperature"]):
            yield ("fuelNoGad", float(uenrich), 0.0, temp, xs, "255 255 150")

    # 2. Gad-bearing Fuel Stream (gad_conc > 0)
    nonzero_gad = gad_concs[~np.isclose(gad_concs, 0.0)]
    for gad_conc, gad_uenrich, (temp, xs) in itertools.product(
        nonzero_gad, axes["gad_uenrich"], axes["temperature"]
    ):
        yield ("fuelYesGad", float(gad_uenrich), float(gad_conc), temp, xs, "150 255 150")


def append_fuel_library_from_axes(
    axes,
    output_file="materials_library4.txt",
    density=-10.3070,
):
    """
    Generate Serpent material library for all standard and Gd-bearing fuel combinations.
    """
    library = {}

    with open(output_file, "a") as f:
        for family, enrichment, gad_conc, temperature, xs, colour in generate_fuel_combos(axes):
            
            name = f"{family}_{gad_conc:.3f}gd_{enrichment:.3f}wt_{temperature}K"

            spec = FuelSpecification(  # type: ignore
                name=name,
                enrichment=enrichment,
                gad_loading=gad_conc,
                density=density,
                temperature=temperature,
                xs=xs,
                rgb=colour,
                burn=1,
                comment=(
                    f"{family}\n"
                    f"{enrichment:.3f} wt-% U235\n"
                    f"{gad_conc:.3f} wt-% Gd2O3\n"
                    f"{temperature} K"
                ),
            )

            material = build_fuel_material(spec)

            # Verify alignment
            assert material.name == name, f"Name mismatch: {material.name} != {name}"

            # Store using the verified object name
            library[material.name] = material

            # Write Serpent card
            f.write(
                f"% {family}\n"
                f"% {enrichment:.3f} wt-% U235\n"
                f"% {gad_conc:.3f} wt-% Gd2O3\n"
                f"% Temperature {temperature} K\n"
            )

            f.write(
                f"mat {material.name} "
                f"{material.density} "
                f"tmp {material.tmp} "
                f"rgb {material.rgb} "
                f"burn {material.burn}\n"
            )

            for nuc in material.nuclides:
                f.write(f"{nuc.zaid:<12} {nuc.frac:.8e}\n")

            f.write("\n")

    return library

In [6]:
np.linspace(0, 12, 121)

array([ 0. ,  0.1,  0.2,  0.3,  0.4,  0.5,  0.6,  0.7,  0.8,  0.9,  1. ,
        1.1,  1.2,  1.3,  1.4,  1.5,  1.6,  1.7,  1.8,  1.9,  2. ,  2.1,
        2.2,  2.3,  2.4,  2.5,  2.6,  2.7,  2.8,  2.9,  3. ,  3.1,  3.2,
        3.3,  3.4,  3.5,  3.6,  3.7,  3.8,  3.9,  4. ,  4.1,  4.2,  4.3,
        4.4,  4.5,  4.6,  4.7,  4.8,  4.9,  5. ,  5.1,  5.2,  5.3,  5.4,
        5.5,  5.6,  5.7,  5.8,  5.9,  6. ,  6.1,  6.2,  6.3,  6.4,  6.5,
        6.6,  6.7,  6.8,  6.9,  7. ,  7.1,  7.2,  7.3,  7.4,  7.5,  7.6,
        7.7,  7.8,  7.9,  8. ,  8.1,  8.2,  8.3,  8.4,  8.5,  8.6,  8.7,
        8.8,  8.9,  9. ,  9.1,  9.2,  9.3,  9.4,  9.5,  9.6,  9.7,  9.8,
        9.9, 10. , 10.1, 10.2, 10.3, 10.4, 10.5, 10.6, 10.7, 10.8, 10.9,
       11. , 11.1, 11.2, 11.3, 11.4, 11.5, 11.6, 11.7, 11.8, 11.9, 12. ])

In [7]:
axes = {  
    "uenrich": np.linspace(0,6,121),  # Standard fuel U-235 wt%
    "gad_uenrich": np.linspace(0,2,81),  # Gad pin U-235 wt% (e.g. 0.8wt)
    "gad_conc": np.linspace(0, 12, 121),    
    "temperature": [
        (300, "03c"),
        (500, "03c"),
        (900, "09c"),
        (950, "09c"),
    ],
}

In [8]:
append_fuel_library_from_axes(axes)

{'fuelNoGad_0.000gd_0.000wt_300K': Material(name='fuelNoGad_0.000gd_0.000wt_300K', density=-10.307, nuclides=[Nuclide(zaid='92235.03c', frac=0.0, comment=None), Nuclide(zaid='92238.03c', frac=0.33333333333333337, comment=None), Nuclide(zaid='8016.03c', frac=0.6666666666666667, comment=None)], tmp=300, moder=None, burn=1, rgb='255 255 150', header_comment='fuelNoGad\n0.000 wt-% U235\n0.000 wt-% Gd2O3\n300 K'),
 'fuelNoGad_0.000gd_0.000wt_500K': Material(name='fuelNoGad_0.000gd_0.000wt_500K', density=-10.307, nuclides=[Nuclide(zaid='92235.03c', frac=0.0, comment=None), Nuclide(zaid='92238.03c', frac=0.33333333333333337, comment=None), Nuclide(zaid='8016.03c', frac=0.6666666666666667, comment=None)], tmp=500, moder=None, burn=1, rgb='255 255 150', header_comment='fuelNoGad\n0.000 wt-% U235\n0.000 wt-% Gd2O3\n500 K'),
 'fuelNoGad_0.000gd_0.000wt_900K': Material(name='fuelNoGad_0.000gd_0.000wt_900K', density=-10.307, nuclides=[Nuclide(zaid='92235.09c', frac=0.0, comment=None), Nuclide(zaid=